# LFW Grad-CAM — 04. Step 2 compression characterization

동일한 `origin_embedding_artifact_uid`의 512D만 PCA-only 또는 PQ-only
정량 runner에 전달합니다. 이 노트북은 PCA/PQ fitting 로직을 복제하지 않고,
runner 결과가 전체 표본·모델 provenance와 fallback-free 조건을 지키는지
검사해 lineage 열을 붙입니다.

현재 전용 PyTorch Step 2 정량 runner가 아직 없으므로 실제 runner 출력
경로가 없으면 의도적으로 중단합니다.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import yaml

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("프로젝트 루트(C:/ronbun)를 찾을 수 없습니다.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs/experiments/step2_pytorch_gradcam.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

EXECUTION = CONFIG["execution"]
MODEL_PROFILE = str(EXECUTION["model_profile"])
MODE = str(EXECUTION["mode"])
DATA_FRACTION = float(EXECUTION["data_fraction"])
SEED = int(EXECUTION["seed"])
EXECUTE_STAGE = bool(EXECUTION["execute_stage"])
WRITE_OUTPUTS = bool(EXECUTION["write_outputs"])
OVERWRITE = bool(EXECUTION["overwrite"])

if MODEL_PROFILE not in CONFIG["models"]["selected_profiles"]:
    raise ValueError(f"선택되지 않은 모델 profile: {MODEL_PROFILE}")
MODEL_NAME = str(CONFIG["models"]["profiles"][MODEL_PROFILE]["family"])
if MODE not in {"dev", "real"}:
    raise ValueError("MODE는 'dev' 또는 'real'이어야 합니다.")
if not 0.0 < DATA_FRACTION <= 1.0:
    raise ValueError("DATA_FRACTION은 (0, 1] 범위여야 합니다.")
if MODE == "real" and DATA_FRACTION != 1.0:
    raise ValueError("real 모드는 DATA_FRACTION=1.0이어야 합니다.")
if WRITE_OUTPUTS and not EXECUTE_STAGE:
    raise ValueError("WRITE_OUTPUTS=True이면 EXECUTE_STAGE도 True여야 합니다.")


In [ ]:
from research.runtime import RunStore, resolve_active_dataset_run

STEP2_RUN_DIR = resolve_active_dataset_run(
    PROJECT_ROOT / CONFIG["run"]["root"],
    dataset_id=str(CONFIG["run"]["dataset_id"]),
    directory_template=str(CONFIG["run"]["dataset_date_dir_template"]),
)
RUN = RunStore.open(STEP2_RUN_DIR)
WORKFLOW_ROOT = (
    STEP2_RUN_DIR / CONFIG["workflow"]["artifact_subdir"]
)

import pandas as pd

from research.evaluation import annotate_compression_lineage
from research.experiments import characterize_step2_compression
from research.explainability.gradcam import read_prepared_population_artifact

PREPARED_ARTIFACT_DIR = WORKFLOW_ROOT / CONFIG["workflow"]["prepared_population_dir"]
SELECTED_MANIFEST_PATH = WORKFLOW_ROOT / CONFIG["workflow"]["selected_manifest_path"]
GALLERY_IDENTITIES_PATH = PROJECT_ROOT / CONFIG["datasets"]["lfw"]["gallery_identities_path"]
UNKNOWN_IDENTITIES_PATH = PROJECT_ROOT / CONFIG["datasets"]["lfw"]["unknown_unknown_identities_path"]
PAIRED_METRICS_OUTPUT_PATH = WORKFLOW_ROOT / CONFIG["workflow"]["paired_metrics_path"]
RETRIEVAL_METRICS_OUTPUT_PATH = WORKFLOW_ROOT / CONFIG["workflow"]["retrieval_metrics_path"]

In [ ]:
if EXECUTE_STAGE:
    required = {
        "prepared": PREPARED_ARTIFACT_DIR,
        "selected": SELECTED_MANIFEST_PATH,
        "gallery_identities": GALLERY_IDENTITIES_PATH,
        "unknown_identities": UNKNOWN_IDENTITIES_PATH,
    }
    missing = [name for name, path in required.items() if not Path(path).exists()]
    if missing:
        raise FileNotFoundError(f"Step 2 압축 입력이 없습니다: {missing}")
    prepared = read_prepared_population_artifact(required["prepared"])
    selected = pd.read_csv(required["selected"])
    read_ids = lambda path: tuple(
        line.strip()
        for line in Path(path).read_text(encoding="utf-8").splitlines()
        if line.strip()
    )
    pca_dimensions = tuple(
        int(value)
        for value in CONFIG["compression"]["families"]["pca"]["dimensions"]
    )
    pq_settings = tuple(
        (int(item["m"]), int(item["nbits"]))
        for item in CONFIG["compression"]["families"]["pq"]["settings"]
    )
    result = characterize_step2_compression(
        prepared,
        selected,
        gallery_identities=read_ids(required["gallery_identities"]),
        unknown_unknown_identities=read_ids(required["unknown_identities"]),
        pca_dimensions=pca_dimensions,
        pq_settings=pq_settings,
        seed=SEED,
        target_fpir=float(CONFIG["evaluation"]["target_fpir"]),
        enrollment_count=int(CONFIG["evaluation"]["lfw_enrollment_count"]),
        calibration_gallery_identities=int(
            CONFIG["evaluation"]["calibration_gallery_identities"]
        ),
        top_k=int(CONFIG["evaluation"]["top_k"]),
    )
    lineage = {
        "extraction_uid": prepared.extraction_uid,
        "dataset_id": prepared.dataset_id,
        "model_uid": prepared.model_uid,
        "origin_embedding_artifact_uid": prepared.origin_embedding_artifact_uid,
    }
    paired = annotate_compression_lineage(result.paired_metrics, **lineage)
    retrieval = annotate_compression_lineage(result.retrieval_metrics, **lineage)
    if "pca_pq" in set(paired["compression_family"].astype(str)):
        raise ValueError("PCA→PQ는 독립 실험군이 아닙니다.")
    for name, frame in (("paired", paired), ("retrieval", retrieval)):
        if frame["origin_fallback_used"].fillna(True).astype(bool).any():
            raise ValueError(f"{name}에 origin fallback 행이 있습니다.")
    compression_summary = result.summary.assign(**lineage)
    if WRITE_OUTPUTS:
        outputs = (
            (PAIRED_METRICS_OUTPUT_PATH, paired),
            (RETRIEVAL_METRICS_OUTPUT_PATH, retrieval),
        )
        for destination, frame in outputs:
            destination = Path(destination).resolve()
            if destination.exists() and not OVERWRITE:
                raise FileExistsError(f"canonical 결과가 이미 있습니다: {destination}")
            destination.parent.mkdir(parents=True, exist_ok=True)
            frame.to_csv(destination, index=False, encoding="utf-8")
else:
    compression_summary = pd.DataFrame(
        [{"status": "not_executed", "reason": "EXECUTE_STAGE=False"}]
    )
compression_summary

Grad-CAM 특징은 임베딩에 이어 붙이거나 PCA/PQ 입력으로 사용하지 않습니다.
두 결과는 다음 단계에서 표 수준으로만 결합합니다.
